[![OpenSD2026 - Belgium](../../../assets/OpenSD_bannerlogo.jpg)](https://www.vub.be/en/event/opensd-summer-school-belgium)

Copyright © 2026 OpenSD2026 contributors. All rights reserved. The materials in this repository are provided for educational use only. See [LICENSE](../../../LICENSE).

# Test-Driven Development with `pytest`

**Test-Driven Development (TDD)** is an approach where we write the **test before writing the implementation**.

The workflow is simple:

1. **Write a test** describing what the code should do.
2. **Run the test** and see it fail.
3. **Write the code** needed to make the test pass.
4. **Refactor** while keeping the tests passing.

## Why is this especially relevant in the era of AI?

AI tools can generate code very quickly, but generated code is not necessarily **correct**.

This makes testing even more important. Instead of asking an AI to generate some code and then trying to understand whether it works, we can first define the **expected behaviour through tests**.

The tests then become a clear specification that both humans and AI-generated code must satisfy:

> **We define what "correct" means first, then generate or write the implementation.**

For scientific and engineering software, this is particularly useful because we often already know physical properties, analytical solutions, limiting cases, or expected outputs that can be turned into tests.

In this notebook, we will apply this idea to a simple **projectile-motion model**: first define the expected physical behaviour, then implement the model that satisfies it.

## Projectile motion

Suppose we need a function called `landing_distance(speed, angle_degrees)` that returns where a ball lands after being shot at a given `speed` and angle `angle_degrees`.  Imagine we do not know the equation yet, but we already have known correct values in [OpenStax University Physics, Figure 4.15](https://openstax.org/books/university-physics-volume-1/pages/4-3-projectile-motion):

| Speed [m/s] | Angle | Range [m] |
| ---: | ---: | ---: |
| 30 | 45° | 91.8 |
| 40 | 45° | 163 |
| 50 | 45° | 255 |
| 50 | 15° | 128 |
| 50 | 75° | 128 |

These values assume level ground and no air resistance. But they will act as (one of many) ground truth. Whatever we do to develop `landing_distance` it should always give these values. Note the values are rounded, so we will allow a 1% difference in the tests.

## Write the test first

Start with one value that we already know.

In [ ]:
import pytest


def test_one_reference_value():
    result = landing_distance(speed=30, angle_degrees=45) # Calculate the landing distance for the given speed and angle
    assert result == pytest.approx(91.8, rel=0.01), f'This is not a correct result; got "{result}", should be "91.8"' # Check if the calculated landing distance is approximately 91.8 with a relative tolerance of 1%, as mentioned in our 'ground truth'

On itself the code above doesn't do anything, it just defines the test. But if this test is ran now, it would fail. Simply because `landing_distance` has not been written yet.

## Write the function

We can now implement the projectile equation and run the same test again.

In [ ]:
from math import radians, sin


def landing_distance(speed, angle_degrees):
    angle = (angle_degrees)
    gravity = 9.81
    return speed**2 * sin(2 * angle) / gravity

Let us run our test:

In [ ]:
test_one_reference_value()

Ooh no! At first run the test fails, clearly we made a mistake in our implementation! 

🔎 **Task**
Have a look back to the function, did we do anything wrong? Fix the function!

<details>
<summary>💡 Need a hint? Click to expand</summary>
The angle in degrees needs to be converted to radians for this function to work.
</details>

Let us redo the test after your fix

In [ ]:
test_one_reference_value()


## Test the other known values

Instead of copying the test five times, use `@pytest.mark.parametrize`. Pytest will report each row as a separate test.

In [ ]:
REFERENCE_CASES = [
    (30, 45, 91.8),
    (40, 45, 163),
    (50, 45, 255),
    (50, 15, 128),
    (50, 75, 128),
]


@pytest.mark.parametrize(
    ("speed", "angle_degrees", "expected"),
    REFERENCE_CASES,
)
def test_openstax_reference(speed, angle_degrees, expected):
    result = landing_distance(speed, angle_degrees)
    assert result == pytest.approx(expected, rel=0.01), f'This is not a correct result; got "{result}", should be "{expected}"'

> **Exercise:** We also know that a vertical launch ($90°$) has no horizontal motion. Add a separate test for this case. Use an absolute tolerance because the expected result is zero.

<details>
<summary><strong>Show answer</strong></summary>

```python
def test_vertical_launch_has_zero_range():
    result = landing_distance(speed=10, angle_degrees=90)
    assert result == pytest.approx(0, abs=1e-12)
```

</details>

In [ ]:
def test_vertical_launch_has_zero_range():
    assert False # Placeholder for the actual test implementation, this will always fail until implemented

### Add more behaviour with parametrization

Now we expand the specification with the remaining reference values. The `@pytest.mark.parametrize` decorator passes each tuple in `REFERENCE_CASES` to the same test function, so pytest reports every case separately.

We also add the known $90°$ limiting case. The loop at the bottom executes the cases in this notebook; pytest performs that expansion automatically in a test file.

## 👷 Engineers aren't programmers

**Now you might think, why can't AI write these tests?!** Well it can, and it will be a good help. But AI will focus initially on basic code checks, e.g. that errors are flagged. It won't always challenge the physics of a problem, or it won't know properly what is the ground truth.

Good practice therefore is to let the test start from physics and engineering. E.g. are you developing a tool to do Finite Element Modeling? Maybe use the analytical results of simple structures as the starting point.

In this day and age 'Test-driven development' really should become a best practice. First define what you expect the code to do in tests. Spend time to really think about this.



## Further developing the function

Let us further develop our function, I want to expand to work for other planets. How far will it shoot on e.g. the moon where gravity is approximately 1.625 m/s2? 

To do so well expand the function `landing_distance` to include an additional keyword argument `gravity`.

**Task** Write a new test that tests the results for various values of the gravitational acceleration.



In [ ]:
def test_at_different_gravity():
    # Write a test to check the landing distance for different gravity values
    assert False # Placeholder for the actual test implementation, this will always fail until implemented

Now update the original function, so this new test passes as well as all previous tests!

In [ ]:
test_one_reference_value()
test_vertical_launch_has_zero_range()
test_at_different_gravity()

# Integrating tests into your package

The cool thing is that you can bake in these tests to your package. This creates a 'safety' measure for packages. It is an important practice to make sure your code always keeps doing what you expect it to do (which is what you set in the tests.)

🔎 **Task** Out of curiosity we can have a look at tests of the py-fatigue package: e.g. focusing on the CycleCounting; [Test histogram](https://github.com/OWI-Lab/py_fatigue/blob/develop/tests/cycle_count/test_histogram.py). This particular test test the cyclecounting against pre-defined (normed) results.

## How to do this? 
The tests can be added into a package by respecting a fixed structure;

The relevant files in the finished example are:

```text
12_test_driven_dev/
├── Awesome_openSD/
│   └── projectile.py
├── tests/
│   └── test_projectile.py
└── 12_test_driven_dev.ipynb
```
To automate the testing, you can simply run the tests from the commandline.

```bash
uv run pytest
or
uv run pytest notebooks/WS1_An_intro_to_python/12_test_driven_dev/tests/test_projectile.py -q
```

Doing this in this session would lead us too far but, we hope we gave you a tast of testing.
